# RAG Bot Test 2: API Connection (Weather Tool)

This notebook is a small, standalone experiment: instead of retrieving answers from local
documents, the model will call a **live public API** to answer a question.

Stack for this notebook:
- Ollama runs the LLM (`llama3.1:8b`) locally, same as the main project
- Ollama's built-in **tool calling** lets the model decide when to call a Python function
- [Open-Meteo](https://open-meteo.com) is used as the API. It's free, requires no API key,
  and returns simple JSON weather data

No documents, no Chroma, no LangChain chain here, just: ask a question, model calls a tool,
tool hits an API, model writes the final answer. This is a first step toward giving the main
RAG bot the ability to reach outside its own documents when useful.


## Step 0: Before running anything

Same requirement as the other notebooks in this project:

1. Ollama is installed and running (llama icon visible in the system tray).
2. The `llama3.1:8b` model is pulled:
   ```
   ollama pull llama3.1:8b
   ```
3. The project's virtual environment is activated.

The next cell checks that Ollama is alive and that the model is installed. Don't continue if
this fails go fix Ollama first.


In [1]:
# --- Sanity check: is Ollama running, and is our model installed? ---
import requests

try:
    resp = requests.get("http://localhost:11434/api/tags", timeout=5)
    resp.raise_for_status()
    installed = [m["name"] for m in resp.json().get("models", [])]

    print("Ollama is running.")
    print("Installed models:", installed)

    needed = "llama3.1:8b"
    base = needed.split(":")[0]
    if any(m.startswith(base) for m in installed):
        print(f"found a model matching '{base}'")
    else:
        print(f"'{base}' not found — run: ollama pull {needed}")
except Exception as e:
    print("Could not reach Ollama at http://localhost:11434")
    print("Make sure the Ollama app is running, then re-run this cell.")
    print("Error:", e)


Ollama is running.
Installed models: ['nomic-embed-text:latest', 'llama3.1:8b', 'qwen3-coder:30b', 'qwen2.5-coder:7b']
found a model matching 'llama3.1'


## Step 1: Install the Python packages

This notebook only needs two extra things beyond the base project: the `ollama` Python client
and `requests` for calling the weather API. `requests` is almost certainly already installed;
`ollama` may not be.

Safe to skip if these are already installed via `requirements.txt`.


In [2]:
# Run once if needed. Safe to skip if already installed.
%pip install -q ollama requests
print("Done. Restart the kernel if this was a fresh install.")


Note: you may need to restart the kernel to use updated packages.
Done. Restart the kernel if this was a fresh install.


## Step 2: Configuration

Same habit as the main notebook: all the knobs live in one place.

- `LLM_MODEL`: the chat model that decides when to call the tool and writes the final answer.
- `WEATHER_API_URL`: the Open-Meteo endpoint. No API key needed.


In [1]:
LLM_MODEL = "llama3.1:8b"
WEATHER_API_URL = "https://api.open-meteo.com/v1/forecast"

print("Config loaded. Using model:", LLM_MODEL)


Config loaded. Using model: llama3.1:8b


## Step 3: Define the tool, a plain Python function

This is the actual API connection. Given a latitude and longitude, it calls Open-Meteo and
returns the current temperature as JSON. Nothing fancy, a normal `requests.get` call.

The docstring matters: it's not just documentation, it's what gets shown to the model later so
it understands what this function does and when to use it.


In [2]:
import requests

def get_current_weather(latitude: float, longitude: float) -> dict:
    """Get the current temperature for a location, given its latitude and longitude."""
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m",
    }
    response = requests.get(WEATHER_API_URL, params=params, timeout=5)
    response.raise_for_status()
    return response.json()

# quick manual test — Groningen, NL
get_current_weather(53.22, 6.57)


{'latitude': 53.211998,
 'longitude': 6.5829997,
 'generationtime_ms': 0.038743019104003906,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 7.0,
 'current_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature_2m': '°C'},
 'current': {'time': '2026-07-07T09:45',
  'interval': 900,
  'temperature_2m': 22.0}}

## Step 4: Describe the tool to the model

Ollama's tool calling works by giving the model a JSON schema describing available functions:
their name, description, and parameters. The model then decides on its own whether a question
needs the tool, and if so, which arguments to call it with.

This schema mirrors the Python function above same name, same parameters.


In [3]:
weather_tool_schema = {
    "type": "function",
    "function": {
        "name": "get_current_weather",
        "description": "Get the current temperature for a location given its latitude and longitude.",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": {"type": "number", "description": "Latitude of the location"},
                "longitude": {"type": "number", "description": "Longitude of the location"},
            },
            "required": ["latitude", "longitude"],
        },
    },
}

print("Tool schema ready.")


Tool schema ready.


## Step 5: Ask a question and let the model call the tool

The flow:
1. Send the question to the model along with the tool schema.
2. If the model responds with a `tool_call`, run the matching Python function with the
   arguments the model extracted.
3. Send the tool's result back to the model as a follow-up message.
4. The model writes the final, natural-language answer.

This is the same request → response → final answer loop used by any tool-calling setup,
just written out explicitly so each step is visible.


In [4]:
import ollama

def ask_with_weather_tool(question: str) -> str:
    messages = [{"role": "user", "content": question}]

    response = ollama.chat(
        model=LLM_MODEL,
        messages=messages,
        tools=[weather_tool_schema],
    )

    tool_calls = response["message"].get("tool_calls")

    if not tool_calls:
        # model answered directly without needing the tool
        return response["message"]["content"]

    # the model wants to call our function
    messages.append(response["message"])

    for call in tool_calls:
        args = call["function"]["arguments"]
        print("Model requested tool call:", call["function"]["name"], args)

        result = get_current_weather(**args)

        messages.append({
            "role": "tool",
            "content": str(result),
        })

    # send the tool result back so the model can phrase the final answer
    follow_up = ollama.chat(model=LLM_MODEL, messages=messages)
    return follow_up["message"]["content"]


question = "What's the current temperature in Groningen, Netherlands?"  # edit this
answer = ask_with_weather_tool(question)
print(answer)


Model requested tool call: get_current_weather {'latitude': 53.2176, 'longitude': 6.5667}
The current temperature in Groningen, Netherlands is 21.5°C (as of July 7th, 2023 at 9:45). Please note that the information may change depending on your location and time zone.


## Step 6: A simple interactive loop

Run this cell to ask multiple questions in a row. Type `quit` or `exit` to stop.

Try both weather questions (which should trigger the tool) and unrelated questions (which
should get answered directly, with no API call at all) to see the model make that decision.


In [ ]:
while True:
    q = input("Ask something (or 'quit'): ").strip()
    if q.lower() in ("quit", "exit"):
        break
    if not q:
        continue
    print(ask_with_weather_tool(q))
    print("-" * 60)


Ask something (or 'quit'):  Whats the weather in Brussels like?


Model requested tool call: get_current_weather {'longitude': 4.35, 'latitude': 50.85}
The current weather in Brussels is mostly cloudy with a temperature of 26.5°C (79.7°F).
------------------------------------------------------------


Ask something (or 'quit'):  How about Paris?


Model requested tool call: get_current_weather {'latitude': 48.8566, 'longitude': 2.3522}
The current weather in Paris is as follows:

* Temperature: 29.9°C (85.8°F)
* Time: July 7th, 2023 at 09:45 UTC
* Elevation: 36 meters above sea level

Please note that this information may not be up-to-date or accurate, and you should check a weather website or app for the most current information.
------------------------------------------------------------


Ask something (or 'quit'):  Berlin?


Model requested tool call: get_current_weather {'longitude': 13.405, 'latitude': 52.52}
You're looking for information about Berlin!

Berlin is the capital and largest city of Germany by both population and area. It is located in northeastern Germany on the banks of the River Spree.

Here are some interesting facts about Berlin:

1. **Rich History**: Berlin has a rich history, from being the capital of Prussia to becoming the center of the Weimar Republic and Nazi Germany during World War II.
2. **Cultural Hub**: Berlin is known for its vibrant cultural scene, with numerous museums, galleries, theaters, and music venues.
3. **Nightlife**: The city has a lively nightlife, with many bars, clubs, and lounges to choose from.
4. **Food**: Berlin is famous for its currywurst (a sausage dish topped with ketchup-based sauce and curry powder) as well as other traditional German cuisine such as schnitzel and sauerbraten.
5. **Architecture**: The city has a mix of architectural styles, from moder

## Next steps

- Add more tools alongside this one (e.g. a currency converter, a joke API) and see how the
  model chooses between them.
- Combine this notebook's tool-calling pattern with the main project's document retriever, so
  the bot can pick between "answer from company docs" and "answer from a live API" depending
  on the question.
- Add error handling for when the API is unreachable or returns unexpected data.
